## user data cleaning/processing notebook

This notebook is used to test data cleaning/processing from user intake [form](https://docs.google.com/forms/d/1Oi52PxcGuTFLCc_kkKUzp_SW7i9ElVBZ6XVfXQQrhqs/edit)


Imporant steps
1. load data
2. handle missing values
3. convert venue types into numbers??
4. clean sports teams list
5. Do users need unique IDs?

In [24]:
import pandas as pd
import regex as re
from thefuzz import process, fuzz

# import for levenstein distance

In [25]:
master_teams = pd.read_csv("../data/master_teams.csv")
NBA_TEAMS = master_teams[master_teams["League"] == "NBA"]["Team Name"].tolist()
CBB_TEAMS = master_teams[master_teams["League"].str.contains("CBB")]["Team Name"].unique().tolist()

In [26]:
test1_df = pd.read_csv("../data/User_Responses_Test2.csv")
test1_df

,Timestamp,"Consent to participate\n\nBy clicking ""Yes"" below, you acknowledge that you have read the above information and voluntarily agree to participate in this survey AND receive weekly email newsletters",Full Name,Email Address\n(Note: This will be used to send your weekly curated matchups newsletter),"When choosing a venue to watch a game, what features are most important to you? \n(Select all that apply)",Which Men's College Basketball (CBB) teams do you follow most closely?,Which Women's College Basketball (CBB) teams do you follow most closely?,Which NBA teams do you follow most closely?,Is there anything else you'd like us to know?
0,3/6/2026 11:23:36,"Yes, I consent to participate and proceed",Test 1,lumandigo6@gmail.com,Traditional Sports Bar Environment (Multiple s...,"University of North Carolina, University of Mi...",University of North Carolina,"Atlanta Hawks, Detroit Pistons",NaN
1,3/6/2026 11:25:08,"Yes, I consent to participate and proceed",Test 2,lundenm@umich.edu,Traditional Sports Bar Environment (Multiple s...,"University of Michigan, University of North Ca...",University of Iowa,"Detroit Pistons, Charlotte Hornets",NaN
2,3/6/2026 11:26:03,"Yes, I consent to participate and proceed",Test 3,lumandigo6@gmail.com,"Quality Beer Selection, Signature Cocktails & ...","Duke University, University of Georgia",NaN,Los Angeles Lakers,NaN
3,3/6/2026 11:27:20,"Yes, I consent to participate and proceed",Test 4,lundenm@umich.edu,Family-Friendly Atmosphere,"Indiana University, Notre Dame","Indiana University, Notre Dame",Indiana Pacers,NaN
4,3/6/2026 11:28:37,"Yes, I consent to participate and proceed",Test 5,lumandigo6@gmail.com,"Quality Beer Selection, High-Quality Food Menu...","University of Arizona, University of North Car...",NaN,"Miami Heat, Los Angeles Lakers",NaN
5,3/6/2026 13:36:03,"Yes, I consent to participate and proceed",Test 6,lumandigo6@gmail.com,Wings,Perdue University,NaN,Detrot Pistens,No
6,3/9/2026 17:05:05,"Yes, I consent to participate and proceed",Tyrone Pettygrue,tpetty@umich.edu,Traditional Sports Bar Environment (Multiple s...,University of Michigan,University of Michigan,Detroit Pistons,NaN
7,3/9/2026 17:08:50,"Yes, I consent to participate and proceed",Alex Arce,aarcejr@umich.edu,Traditional Sports Bar Environment (Multiple s...,1. University of Michigan \n2. University of C...,1. University of Michigan\n2. California State...,1. Los Angeles Lakers\n2. San Antonio Spurs\n3...,On the top 3 list show if it should be inline ...


In [27]:
test1_df.columns = ["timestamp", "consent", "fullname", "email", "venueFeatures", "MCBB", "WCBB", "NBA", "additionalComments"]
test1_df

,timestamp,consent,fullname,email,venueFeatures,MCBB,WCBB,NBA,additionalComments
0,3/6/2026 11:23:36,"Yes, I consent to participate and proceed",Test 1,lumandigo6@gmail.com,Traditional Sports Bar Environment (Multiple s...,"University of North Carolina, University of Mi...",University of North Carolina,"Atlanta Hawks, Detroit Pistons",NaN
1,3/6/2026 11:25:08,"Yes, I consent to participate and proceed",Test 2,lundenm@umich.edu,Traditional Sports Bar Environment (Multiple s...,"University of Michigan, University of North Ca...",University of Iowa,"Detroit Pistons, Charlotte Hornets",NaN
2,3/6/2026 11:26:03,"Yes, I consent to participate and proceed",Test 3,lumandigo6@gmail.com,"Quality Beer Selection, Signature Cocktails & ...","Duke University, University of Georgia",NaN,Los Angeles Lakers,NaN
3,3/6/2026 11:27:20,"Yes, I consent to participate and proceed",Test 4,lundenm@umich.edu,Family-Friendly Atmosphere,"Indiana University, Notre Dame","Indiana University, Notre Dame",Indiana Pacers,NaN
4,3/6/2026 11:28:37,"Yes, I consent to participate and proceed",Test 5,lumandigo6@gmail.com,"Quality Beer Selection, High-Quality Food Menu...","University of Arizona, University of North Car...",NaN,"Miami Heat, Los Angeles Lakers",NaN
5,3/6/2026 13:36:03,"Yes, I consent to participate and proceed",Test 6,lumandigo6@gmail.com,Wings,Perdue University,NaN,Detrot Pistens,No
6,3/9/2026 17:05:05,"Yes, I consent to participate and proceed",Tyrone Pettygrue,tpetty@umich.edu,Traditional Sports Bar Environment (Multiple s...,University of Michigan,University of Michigan,Detroit Pistons,NaN
7,3/9/2026 17:08:50,"Yes, I consent to participate and proceed",Alex Arce,aarcejr@umich.edu,Traditional Sports Bar Environment (Multiple s...,1. University of Michigan \n2. University of C...,1. University of Michigan\n2. California State...,1. Los Angeles Lakers\n2. San Antonio Spurs\n3...,On the top 3 list show if it should be inline ...


In [28]:
POSSIBLE_FEATURES = [
    "Traditional Sports Bar Environment (Multiple screens/high energy)",
    "Quality Beer Selection",
    "Signature Cocktails & Spirits",
    "High-Quality Food Menu (Beyond standard pub fare)",
    "Family-Friendly Atmosphere"
]

def convert_venuesFeatures_to_vectors(venueFeatures, possible_features):
    """Convert the venue features string into a vector of binary features.

    Possible features include:
    - Traditional Sports Bar Environment (Multiple screens/high energy)
    - Quality Beer Selection
    - Signature Cocktails & Spirits
    - High-Quality Food Menu (Beyond standard pub fare)
    -Family-Friendly Atmosphere

    If a user submits a new feature, then print message to consider adding it to the possible features list.


    """
    # Initialize the vector with zeros
    feature_vector = [0] * len(possible_features)

    # Split the input string into individual features
    features = re.split(r',\s*', venueFeatures)

    # Check each feature against the possible features and set the corresponding index to 1
    for feature in features:
        feature = feature.strip()  # Remove leading/trailing whitespace
        if feature in possible_features:
            index = possible_features.index(feature)
            feature_vector[index] = 1
        if feature not in possible_features:
            print(f"Feature '{feature}' not recognized. Consider adding it to the possible features list.")

    return feature_vector

In [29]:
test1_df["venueFeatureVector"] = test1_df["venueFeatures"].apply(lambda x: convert_venuesFeatures_to_vectors(x, POSSIBLE_FEATURES))
test1_df

Feature 'Wings' not recognized. Consider adding it to the possible features list.


,timestamp,consent,fullname,email,venueFeatures,MCBB,WCBB,NBA,additionalComments,venueFeatureVector
0,3/6/2026 11:23:36,"Yes, I consent to participate and proceed",Test 1,lumandigo6@gmail.com,Traditional Sports Bar Environment (Multiple s...,"University of North Carolina, University of Mi...",University of North Carolina,"Atlanta Hawks, Detroit Pistons",NaN,"[1, 1, 0, 0, 0]"
1,3/6/2026 11:25:08,"Yes, I consent to participate and proceed",Test 2,lundenm@umich.edu,Traditional Sports Bar Environment (Multiple s...,"University of Michigan, University of North Ca...",University of Iowa,"Detroit Pistons, Charlotte Hornets",NaN,"[1, 1, 0, 1, 0]"
2,3/6/2026 11:26:03,"Yes, I consent to participate and proceed",Test 3,lumandigo6@gmail.com,"Quality Beer Selection, Signature Cocktails & ...","Duke University, University of Georgia",NaN,Los Angeles Lakers,NaN,"[0, 1, 1, 0, 0]"
3,3/6/2026 11:27:20,"Yes, I consent to participate and proceed",Test 4,lundenm@umich.edu,Family-Friendly Atmosphere,"Indiana University, Notre Dame","Indiana University, Notre Dame",Indiana Pacers,NaN,"[0, 0, 0, 0, 1]"
4,3/6/2026 11:28:37,"Yes, I consent to participate and proceed",Test 5,lumandigo6@gmail.com,"Quality Beer Selection, High-Quality Food Menu...","University of Arizona, University of North Car...",NaN,"Miami Heat, Los Angeles Lakers",NaN,"[0, 1, 0, 1, 0]"
5,3/6/2026 13:36:03,"Yes, I consent to participate and proceed",Test 6,lumandigo6@gmail.com,Wings,Perdue University,NaN,Detrot Pistens,No,"[0, 0, 0, 0, 0]"
6,3/9/2026 17:05:05,"Yes, I consent to participate and proceed",Tyrone Pettygrue,tpetty@umich.edu,Traditional Sports Bar Environment (Multiple s...,University of Michigan,University of Michigan,Detroit Pistons,NaN,"[1, 0, 1, 1, 0]"
7,3/9/2026 17:08:50,"Yes, I consent to participate and proceed",Alex Arce,aarcejr@umich.edu,Traditional Sports Bar Environment (Multiple s...,1. University of Michigan \n2. University of C...,1. University of Michigan\n2. California State...,1. Los Angeles Lakers\n2. San Antonio Spurs\n3...,On the top 3 list show if it should be inline ...,"[1, 0, 1, 0, 0]"


In [30]:
def clean_email(email):
    # Convert to lowercase
    email = email.lower()
    
    # Remove leading and trailing whitespace
    email = email.strip()
    
    # Remove any characters that are not allowed in an email address
    email = re.sub(r'[^\w\.\-@]+', '', email)
    
    return email

test1_df["email"] = test1_df["email"].apply(clean_email)
test1_df

,timestamp,consent,fullname,email,venueFeatures,MCBB,WCBB,NBA,additionalComments,venueFeatureVector
0,3/6/2026 11:23:36,"Yes, I consent to participate and proceed",Test 1,lumandigo6@gmail.com,Traditional Sports Bar Environment (Multiple s...,"University of North Carolina, University of Mi...",University of North Carolina,"Atlanta Hawks, Detroit Pistons",NaN,"[1, 1, 0, 0, 0]"
1,3/6/2026 11:25:08,"Yes, I consent to participate and proceed",Test 2,lundenm@umich.edu,Traditional Sports Bar Environment (Multiple s...,"University of Michigan, University of North Ca...",University of Iowa,"Detroit Pistons, Charlotte Hornets",NaN,"[1, 1, 0, 1, 0]"
2,3/6/2026 11:26:03,"Yes, I consent to participate and proceed",Test 3,lumandigo6@gmail.com,"Quality Beer Selection, Signature Cocktails & ...","Duke University, University of Georgia",NaN,Los Angeles Lakers,NaN,"[0, 1, 1, 0, 0]"
3,3/6/2026 11:27:20,"Yes, I consent to participate and proceed",Test 4,lundenm@umich.edu,Family-Friendly Atmosphere,"Indiana University, Notre Dame","Indiana University, Notre Dame",Indiana Pacers,NaN,"[0, 0, 0, 0, 1]"
4,3/6/2026 11:28:37,"Yes, I consent to participate and proceed",Test 5,lumandigo6@gmail.com,"Quality Beer Selection, High-Quality Food Menu...","University of Arizona, University of North Car...",NaN,"Miami Heat, Los Angeles Lakers",NaN,"[0, 1, 0, 1, 0]"
5,3/6/2026 13:36:03,"Yes, I consent to participate and proceed",Test 6,lumandigo6@gmail.com,Wings,Perdue University,NaN,Detrot Pistens,No,"[0, 0, 0, 0, 0]"
6,3/9/2026 17:05:05,"Yes, I consent to participate and proceed",Tyrone Pettygrue,tpetty@umich.edu,Traditional Sports Bar Environment (Multiple s...,University of Michigan,University of Michigan,Detroit Pistons,NaN,"[1, 0, 1, 1, 0]"
7,3/9/2026 17:08:50,"Yes, I consent to participate and proceed",Alex Arce,aarcejr@umich.edu,Traditional Sports Bar Environment (Multiple s...,1. University of Michigan \n2. University of C...,1. University of Michigan\n2. California State...,1. Los Angeles Lakers\n2. San Antonio Spurs\n3...,On the top 3 list show if it should be inline ...,"[1, 0, 1, 0, 0]"


In [31]:
import pandas as pd
import re

CORRECTION_MAP = {
    "um": "University of Michigan",
    "michigan": "University of Michigan",
    "wolverines": "University of Michigan",
    "msu": "Michigan State University",
    "unc": "University of North Carolina",
    "uconn": "University of Connecticut",
    "ucla": "University of California, Los Angeles",
    "lsu": "Louisiana State University",
    "lakers": "Los Angeles Lakers",
    "knicks": "New York Knicks",
    
    # These catch the schools if their commas were stripped during the input split
    "university of california los angeles": "University of California, Los Angeles",
    "california state university fullerton": "California State University, Fullerton"
}

def clean_basketball_survey_data(df):
    """
    Cleans team names, handles numbered lists, and returns a semicolon-separated string.
    """
    mbb_col = "MCBB"
    wbb_col = "WCBB"
    nba_col = "NBA"
    
    cols_to_clean = [mbb_col, wbb_col, nba_col]
    
    def process_response(response):
        if pd.isna(response) or str(response).strip() == "":
            return None
        
        response = str(response)
        
        # 1. Strip out numbered lists (e.g., "1. ", "2)", "3 - ")
        response = re.sub(r'\b\d+[\.\)\-]\s*', '', response)
        
        # 2. Split the teams based on how the user formatted their input
        if '\n' in response:
            # If they used newlines, split by newlines (protects all commas naturally)
            raw_teams = [team.strip() for team in response.split('\n')]
        else:
            # If they typed on one line, temporarily remove UC/CSU commas so we can split safely
            response = re.sub(r'(?i)(California|University),\s+(Los Angeles|Fullerton|Irvine|Davis|Berkeley|San Diego|Santa Barbara|Santa Cruz|Riverside)', r'\1 \2', response)
            
            # Standardize delimiters to commas, then split
            response = re.sub(r'(?i)\s+and\s+|\s*&\s*|\s*/\s*|\s*;\s*', ',', response)
            raw_teams = [team.strip() for team in response.split(',')]
        
        cleaned_teams = []
        for team in raw_teams:
            if not team: 
                continue
            
            # 3. Apply corrections or standardize casing
            lower_team = team.lower()
            if lower_team in CORRECTION_MAP:
                cleaned_teams.append(CORRECTION_MAP[lower_team])
            else:
                cleaned_teams.append(team.title())
                
        # 4. Return as a SAFE, semicolon-separated string!
        return "; ".join(cleaned_teams)

    for col in cols_to_clean:
        if col in df.columns:
            df[col] = df[col].apply(process_response)
            
    return df

# --- Usage Example ---
test1_df = clean_basketball_survey_data(test1_df)

In [32]:
test1_df.iloc[7, 6]

'University Of Michigan; California State University, Fullerton; University Of California, Los Angeles'

In [33]:
def apply_fuzzy_matching(df, master_team_lists, threshold=85):
    """
    Applies fuzzy matching to correct misspelled team names.
    
    Parameters:
    - df: The pandas DataFrame containing the cleaned columns.
    - master_team_lists: A dictionary mapping column names to a list of valid teams.
    - threshold: The minimum similarity score (0-100) required to accept a match.
    """
    
    def fuzzy_match_list(teams_string, valid_teams):
        # Handle empty/NaN values
        if pd.isna(teams_string) or str(teams_string).strip() == "":
            return None
            
        # Split the semicolon-separated string back into a list
        teams = [t.strip() for t in str(teams_string).split(";")]
        
        matched_teams = []
        for team in teams:
            # Skip if the team name is already perfectly in the master list
            if team in valid_teams:
                matched_teams.append(team)
                continue
                
            # Use thefuzz to find the closest match
            # extractOne returns a tuple: (Best Match String, Score, Index)
            best_match, score = process.extractOne(
                team, 
                valid_teams, 
                scorer=fuzz.token_sort_ratio # Good for handling out-of-order words
            )
            
            # If the score meets our confidence threshold, use the corrected name
            if score >= threshold:
                matched_teams.append(best_match)
            else:
                # If the score is too low, keep the original (or flag it for manual review)
                matched_teams.append(team)
                print(f"Team '{team}' did not match any valid team with sufficient confidence. Consider reviewing this entry.")
                
        return "; ".join(matched_teams)

    # Apply to each column that has a corresponding master list
    for col, valid_teams in master_team_lists.items():
        if col in df.columns:
            df[col] = df[col].apply(lambda x: fuzzy_match_list(x, valid_teams))
            
    return df

# 2. Map columns to their respective master lists
column_to_master_list_map = {
    "NBA": NBA_TEAMS,
    "CBB": CBB_TEAMS,
    "WCBB": CBB_TEAMS
}

# 3. Apply the function to your already-cleaned dataframe
final_df = apply_fuzzy_matching(test1_df, column_to_master_list_map, threshold=85)

Team 'California State University, Fullerton' did not match any valid team with sufficient confidence. Consider reviewing this entry.


In [34]:
final_df

,timestamp,consent,fullname,email,venueFeatures,MCBB,WCBB,NBA,additionalComments,venueFeatureVector
0,3/6/2026 11:23:36,"Yes, I consent to participate and proceed",Test 1,lumandigo6@gmail.com,Traditional Sports Bar Environment (Multiple s...,University Of North Carolina; University Of Mi...,University of North Carolina,Atlanta Hawks; Detroit Pistons,NaN,"[1, 1, 0, 0, 0]"
1,3/6/2026 11:25:08,"Yes, I consent to participate and proceed",Test 2,lundenm@umich.edu,Traditional Sports Bar Environment (Multiple s...,University Of Michigan; University Of North Ca...,University of Iowa,Detroit Pistons; Charlotte Hornets,NaN,"[1, 1, 0, 1, 0]"
2,3/6/2026 11:26:03,"Yes, I consent to participate and proceed",Test 3,lumandigo6@gmail.com,"Quality Beer Selection, Signature Cocktails & ...",Duke University; University Of Georgia,NaN,Los Angeles Lakers,NaN,"[0, 1, 1, 0, 0]"
3,3/6/2026 11:27:20,"Yes, I consent to participate and proceed",Test 4,lundenm@umich.edu,Family-Friendly Atmosphere,Indiana University; Notre Dame,Indiana University; Notre Dame,Indiana Pacers,NaN,"[0, 0, 0, 0, 1]"
4,3/6/2026 11:28:37,"Yes, I consent to participate and proceed",Test 5,lumandigo6@gmail.com,"Quality Beer Selection, High-Quality Food Menu...",University Of Arizona; University Of North Car...,NaN,Miami Heat; Los Angeles Lakers,NaN,"[0, 1, 0, 1, 0]"
5,3/6/2026 13:36:03,"Yes, I consent to participate and proceed",Test 6,lumandigo6@gmail.com,Wings,Perdue University,NaN,Detroit Pistons,No,"[0, 0, 0, 0, 0]"
6,3/9/2026 17:05:05,"Yes, I consent to participate and proceed",Tyrone Pettygrue,tpetty@umich.edu,Traditional Sports Bar Environment (Multiple s...,University Of Michigan,University of Michigan,Detroit Pistons,NaN,"[1, 0, 1, 1, 0]"
7,3/9/2026 17:08:50,"Yes, I consent to participate and proceed",Alex Arce,aarcejr@umich.edu,Traditional Sports Bar Environment (Multiple s...,University Of Michigan; University Of Californ...,University of Michigan; California State Unive...,Los Angeles Lakers; San Antonio Spurs; Detroit...,On the top 3 list show if it should be inline ...,"[1, 0, 1, 0, 0]"
